In [1]:
# Hücre 0: hazırlık. Not defterini her açtığınızda ilk bunu çalıştırın.
import os, glob, json, hashlib, shutil
from pathlib import Path

In [4]:
# Not defteri gol klasörünün yanındaysa bu satıra dokunmayın.
CALISMA = Path.cwd()
# Değilse gol klasörünün İÇİNDE durduğu klasörü yazın, örneğin:
# CALISMA = Path(r"bigdata")

os.chdir(CALISMA)
print("Calisma klasoru :", os.getcwd())
if not os.path.isdir("gol_temiz"):
    raise FileNotFoundError(f"{CALISMA} icinde gol klasoru yok")

# Hücre 1: gölde en büyük dosyayı bul
adaylar = glob.glob("gol_temiz/yil=*/*.parquet")
if not adaylar:
    raise FileNotFoundError("gol_temiz/yil=*/ altinda parquet dosyasi yok")

KAYNAK = max(adaylar, key=os.path.getsize)
BOYUT = os.path.getsize(KAYNAK)

print("Dosya :", KAYNAK)
print(f"Boyut : {BOYUT / 1024**2:.2f} MB")

Calisma klasoru : c:\Users\yusuf\Desktop\bigdata
Dosya : gol_temiz\yil=2020\bolum.parquet
Boyut : 358.25 MB


In [5]:
# Hücre 2: dosyayı parçalara kes (KAYNAK Hücre 1'den gelir)
PARCA_BOYUTU = 4 * 1024 * 1024        # 4 MB

parcalar = []
with open(KAYNAK, "rb") as f:
    i = 0
    while True:
        veri = f.read(PARCA_BOYUTU)
        if not veri:
            break
        parcalar.append((f"parca_{i:04d}", veri))
        i += 1

print(f"Parca sayisi : {len(parcalar)}")
print(f"Son parca    : {len(parcalar[-1][1]) / 1024**2:.2f} MB")

Parca sayisi : 90
Son parca    : 2.25 MB


In [6]:
# Hücre 3: parçaları üç makineye dağıt, kataloğu yaz
# (parcalar Hücre 2'den, KAYNAK Hücre 1'den gelir)
MAKINELER = ["makine1", "makine2", "makine3"]
KOPYA = 2

for m in MAKINELER:
    shutil.rmtree(m, ignore_errors=True)
    os.makedirs(m)

katalog = {}
for sira, (ad, veri) in enumerate(parcalar):
    yerler = []
    for k in range(KOPYA):
        hedef = MAKINELER[(sira + k) % len(MAKINELER)]
        with open(f"{hedef}/{ad}", "wb") as f:
            f.write(veri)
        yerler.append(hedef)
    katalog[ad] = {"yerler": yerler, "boyut": len(veri)}

with open("katalog.json", "w") as f:
    json.dump({"kaynak": KAYNAK, "parca_boyutu": PARCA_BOYUTU,
               "kopya": KOPYA, "parcalar": katalog}, f, indent=2)
print("Dagitim tamam, katalog.json yazildi")

Dagitim tamam, katalog.json yazildi


In [7]:
# Hücre 4: hangi makinede ne var, ne kadar yer tuttu
for m in MAKINELER:
    dosyalar = sorted(os.listdir(m))
    numaralar = [int(x.split("_")[1]) for x in dosyalar]
    yer = sum(os.path.getsize(f"{m}/{x}") for x in dosyalar)
    print(f"{m}: {numaralar}  ({yer / 1024**2:.1f} MB)")

toplam = sum(sum(os.path.getsize(f"{m}/{x}") for x in os.listdir(m))
             for m in MAKINELER)
print(f"Diskte toplam : {toplam / 1024**2:.1f} MB")
print(f"Kac kat yer   : {toplam / BOYUT:.2f}")


makine1: [0, 2, 3, 5, 6, 8, 9, 11, 12, 14, 15, 17, 18, 20, 21, 23, 24, 26, 27, 29, 30, 32, 33, 35, 36, 38, 39, 41, 42, 44, 45, 47, 48, 50, 51, 53, 54, 56, 57, 59, 60, 62, 63, 65, 66, 68, 69, 71, 72, 74, 75, 77, 78, 80, 81, 83, 84, 86, 87, 89]  (238.3 MB)
makine2: [0, 1, 3, 4, 6, 7, 9, 10, 12, 13, 15, 16, 18, 19, 21, 22, 24, 25, 27, 28, 30, 31, 33, 34, 36, 37, 39, 40, 42, 43, 45, 46, 48, 49, 51, 52, 54, 55, 57, 58, 60, 61, 63, 64, 66, 67, 69, 70, 72, 73, 75, 76, 78, 79, 81, 82, 84, 85, 87, 88]  (240.0 MB)
makine3: [1, 2, 4, 5, 7, 8, 10, 11, 13, 14, 16, 17, 19, 20, 22, 23, 25, 26, 28, 29, 31, 32, 34, 35, 37, 38, 40, 41, 43, 44, 46, 47, 49, 50, 52, 53, 55, 56, 58, 59, 61, 62, 64, 65, 67, 68, 70, 71, 73, 74, 76, 77, 79, 80, 82, 83, 85, 86, 88, 89]  (238.3 MB)
Diskte toplam : 716.5 MB
Kac kat yer   : 2.00
